# 两阶段训练：去噪 + 反卷积 (完整复现官方 TensorFlow 实现)

本 notebook 使用 PyTorch 完全复现官方 TensorFlow 实现的所有功能：
- **两阶段 UNet**：去噪 → 反卷积 级联
- **完整损失函数**：PSF 重建 + TV + Hessian(4 项) + Laplace + L1
- **边界处理**：insert_xy padding + 裁剪
- **训练配置**：iterations、学习率衰减、混合精度
- **日志和验证**：TensorBoard、定期验证和测试

参考：官方 TensorFlow 实现 `train_two_stage.py`

## 1. 导入依赖

In [9]:
import os
import math
import time
import datetime
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
import cv2

print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用：{torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.4.0+cu121
CUDA 可用：True


## 2. 工具函数 (完全对齐官方)

In [10]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def percentile_norm(x, pmin=0.1, pmax=99.9, eps=1e-8):
    """百分位归一化 (对齐官方 prctile_norm)"""
    lo = np.percentile(x, pmin)
    hi = np.percentile(x, pmax)
    return np.clip((x - lo) / max(hi - lo, eps), 0.0, 1.0)

def load_image(path):
    """加载图像为 float32 数组"""
    arr = np.array(Image.open(path)).astype(np.float32)
    return arr.mean(axis=2) if arr.ndim == 3 else arr

def save_image(path, x):
    """保存为 16-bit TIFF (对齐官方)"""
    x = np.clip(x / max(x.max(), 1e-8), 0.0, 1.0)
    Image.fromarray((x * 65535.0).astype(np.uint16)).save(path)

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def reflect_pad(x, multiple=16, margin=16):
    """反射填充"""
    h, w = x.shape
    new_h = int(math.ceil(h / multiple) * multiple)
    new_w = int(math.ceil(w / multiple) * multiple)
    pad_h, pad_w = max(new_h - h, 0), max(new_w - w, 0)
    top, bottom = margin + pad_h // 2, margin + pad_h - pad_h // 2
    left, right = margin + pad_w // 2, margin + pad_w - pad_w // 2
    return np.pad(x, ((top, bottom), (left, right)), mode="reflect"), (top, bottom, left, right)

def crop_pad(x, padinfo, scale=1):
    """裁剪填充"""
    top, bottom, left, right = [p * scale for p in padinfo]
    h, w = x.shape
    return x[top:h - bottom, left:w - right]

## 3. PSF 加载 (对齐官方)

In [11]:
def gaussian_psf(size=25, sigma=2.0):
    """生成高斯 PSF"""
    if size % 2 == 0:
        size += 1
    r = size // 2
    y, x = np.mgrid[-r:r + 1, -r:r + 1]
    psf = np.exp(-(x ** 2 / (2 * sigma ** 2) + y ** 2 / (2 * sigma ** 2)))
    psf = psf.astype(np.float32)
    psf /= np.maximum(psf.sum(), 1e-8)
    return psf

def load_psf_from_file(psf_path, size=25):
    """从文件加载真实 PSF (不对图像做 percentile_norm!)"""
    psf = load_image(psf_path)
    
    h, w = psf.shape
    center_y, center_x = h // 2, w // 2
    half = size // 2
    
    top = max(0, center_y - half)
    bottom = min(h, center_y + half + 1)
    left = max(0, center_x - half)
    right = min(w, center_x + half + 1)
    
    psf = psf[top:bottom, left:right]
    
    if psf.shape[0] < size or psf.shape[1] < size:
        pad_h = size - psf.shape[0]
        pad_w = size - psf.shape[1]
        psf = np.pad(psf, ((0, pad_h), (0, pad_w)), mode='constant')
    
    psf /= np.maximum(psf.sum(), 1e-8)
    return psf

def make_psf_tensor(psf_path, size=25, device=None):
    """创建 PSF 张量"""
    psf = load_psf_from_file(psf_path, size)
    psf_t = torch.from_numpy(psf)[None, None].to(device)
    return psf_t / psf_t.sum().clamp_min(1e-8)

## 4. 模型定义 (两阶段 UNet)

In [12]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, n_conv=3):
        super().__init__()
        layers = []
        for i in range(n_conv):
            layers.append(nn.Conv2d(in_ch if i == 0 else out_ch, out_ch, 3, padding=1))
            layers.append(nn.ReLU(inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class Encoder(nn.Module):
    def __init__(self, in_ch=1, base_ch=32, depth=4, n_conv=3):
        super().__init__()
        self.blocks = nn.ModuleList()
        self.pools = nn.ModuleList()
        ch = in_ch
        for i in range(depth):
            out_ch = base_ch * (2 ** i)
            self.blocks.append(ConvBlock(ch, out_ch, n_conv))
            self.pools.append(nn.MaxPool2d(2))
            ch = out_ch
        self.out_ch = ch

    def forward(self, x):
        skips = []
        for blk, pool in zip(self.blocks, self.pools):
            x = blk(x)
            skips.append(x)
            x = pool(x)
        return x, skips


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, n_conv=3):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        layers = [nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True)]
        cur = out_ch
        for _ in range(n_conv - 1):
            next_ch = max(out_ch // 2, 1)
            layers.extend([nn.Conv2d(cur, next_ch, 3, padding=1), nn.ReLU(inplace=True)])
            cur = next_ch
        self.conv = nn.Sequential(*layers)
        self.out_ch = cur

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="nearest")
        return self.conv(torch.cat([x, skip], dim=1))


class UNetStage(nn.Module):
    def __init__(self, in_ch=1, base_ch=32, depth=4, n_conv=3):
        super().__init__()
        self.encoder = Encoder(in_ch, base_ch, depth, n_conv)
        mid_ch = self.encoder.out_ch * 2
        self.mid = nn.Sequential(
            nn.Conv2d(self.encoder.out_ch, mid_ch, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, self.encoder.out_ch, 3, padding=1), nn.ReLU(inplace=True),
        )
        self.decoders = nn.ModuleList()
        cur_ch = self.encoder.out_ch
        for i in reversed(range(depth)):
            skip_ch = base_ch * (2 ** i)
            block = DecoderBlock(cur_ch, skip_ch, skip_ch, n_conv)
            self.decoders.append(block)
            cur_ch = block.out_ch
        self.out_ch = cur_ch

    def forward(self, x):
        x, skips = self.encoder(x)
        x = self.mid(x)
        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)
        return x


class DenoiseStage(nn.Module):
    """阶段 1：去噪"""
    def __init__(self, base_ch=32, depth=4, n_conv=3):
        super().__init__()
        self.encoder = UNetStage(1, base_ch, depth, n_conv)
        self.out = nn.Conv2d(self.encoder.out_ch, 1, 3, padding=1)

    def forward(self, x):
        feat = self.encoder(x)
        return F.relu(self.out(feat))


class DeconvStage(nn.Module):
    """阶段 2：反卷积"""
    def __init__(self, base_ch=32, depth=4, n_conv=3, upsample=True):
        super().__init__()
        self.upsample = upsample
        self.stage2 = UNetStage(1, base_ch, depth, n_conv)
        self.refine1 = nn.Conv2d(self.stage2.out_ch, 128, 3, padding=1)
        self.refine2 = nn.Conv2d(128, 128, 3, padding=1)
        self.out2 = nn.Conv2d(128, 1, 3, padding=1)

    def forward(self, x):
        f2 = self.stage2(x)
        if self.upsample:
            f2 = F.interpolate(f2, scale_factor=2, mode="nearest")
        deconv = F.relu(self.out2(F.relu(self.refine2(F.relu(self.refine1(f2))))))
        return deconv


class TwoStageUNet(nn.Module):
    """
    两阶段 UNet (完全对齐官方 TensorFlow 实现)
    
    输入 → DenoiseStage → 去噪图像 → DeconvStage → 反卷积结果
    """
    def __init__(self, base_ch=32, depth=4, n_conv=3, upsample=True):
        super().__init__()
        self.upsample = upsample
        self.denoise = DenoiseStage(base_ch, depth, n_conv)
        self.deconv = DeconvStage(base_ch, depth, n_conv, upsample)
    
    def forward(self, x):
        denoised = self.denoise(x)
        deconv = self.deconv(denoised)
        return denoised, deconv

## 5. 完整损失函数 (对齐官方)

In [13]:
class HessianLoss(nn.Module):
    """
    Hessian 正则化损失 (4 项：xx, yy, xy, yx)
    完全对齐官方 TensorFlow 实现
    """
    def forward(self, x):
        # 二阶导数
        dxx = x[:, :, :, :-2] - 2 * x[:, :, :, 1:-1] + x[:, :, :, 2:]
        dyy = x[:, :, :-2, :] - 2 * x[:, :, 1:-1, :] + x[:, :, 2:, :]
        
        # 混合偏导 (4 项)
        dxy = x[:, :, 1:, 1:] - x[:, :, :-1, 1:] - x[:, :, 1:, :-1] + x[:, :, :-1, :-1]
        dyx = x[:, :, 1:, 1:] - x[:, :, 1:, :-1] - x[:, :, :-1, 1:] + x[:, :, :-1, :-1]
        
        return (dxx.pow(2).mean() + dyy.pow(2).mean() + 
                dxy.pow(2).mean() + dyx.pow(2).mean()) / 4.0


class TVLoss(nn.Module):
    """Total Variation 正则"""
    def forward(self, x):
        dx = x[:, :, :, 1:] - x[:, :, :, :-1]
        dy = x[:, :, 1:, :] - x[:, :, :-1, :]
        return dx.pow(2).mean() + dy.pow(2).mean()


class LaplaceLoss(nn.Module):
    """
    Laplace 正则化 (官方实现)
    3x3 卷积核：[[1,1,1],[1,-8,1],[1,1,1]]
    """
    def __init__(self):
        super().__init__()
        kernel = torch.tensor([[1, 1, 1],
                               [1,-8, 1],
                               [1, 1, 1]], dtype=torch.float32)
        self.register_buffer('kernel', kernel.view(1, 1, 3, 3))
    
    def forward(self, x):
        lap = F.conv2d(x, self.kernel, padding=1)
        return lap.pow(2).mean()


class PSFLoss(nn.Module):
    """
    PSF 损失函数 (完全对齐官方 TensorFlow)
    
    loss = PSF_reconstruction + TV_weight*TV + Hess_weight*Hessian + 
           laplace_weight*Laplace + l1_rate*L1
    """
    def __init__(self, psf, TV_weight=0, Hess_weight=0.02, laplace_weight=0,
                 l1_rate=0, mse_flag=False, upsample_flag=True, 
                 insert_xy=16, deconv_flag=True):
        super().__init__()
        self.register_buffer("psf", psf)
        self.TV_weight = TV_weight
        self.Hess_weight = Hess_weight
        self.laplace_weight = laplace_weight
        self.l1_rate = l1_rate
        self.mse_flag = mse_flag
        self.upsample_flag = upsample_flag
        self.insert_xy = insert_xy
        self.deconv_flag = deconv_flag
        
        if TV_weight > 0:
            self.tv_loss = TVLoss()
        if Hess_weight > 0:
            self.hess_loss = HessianLoss()
        if laplace_weight > 0:
            self.laplace_loss = LaplaceLoss()
    
    def forward(self, y_true, y_pred):
        B, C, H, W = y_pred.shape
        
        # PSF 卷积
        if self.deconv_flag:
            k = self.psf.shape[-1]
            pad = k // 2
            y_conv = F.conv2d(F.pad(y_pred, (pad, pad, pad, pad), mode="reflect"),
                            self.psf, padding=0)
        else:
            y_conv = y_pred
        
        # 下采样到原始尺寸
        if self.upsample_flag:
            y_conv = F.interpolate(y_conv, size=(H, W), mode="bilinear", 
                                  align_corners=False)
        
        # 裁剪边界 (insert_xy)
        if self.insert_xy > 0:
            y_conv = y_conv[..., self.insert_xy:-self.insert_xy, 
                           self.insert_xy:-self.insert_xy]
            y_true = y_true[..., self.insert_xy:-self.insert_xy, 
                           self.insert_xy:-self.insert_xy]
        
        # 重建损失
        if self.mse_flag:
            psf_loss = F.mse_loss(y_conv, y_true)
        else:
            psf_loss = F.l1_loss(y_conv, y_true)
        
        # TV 损失
        TV_loss = 0
        if self.TV_weight > 0:
            TV_loss = self.tv_loss(y_pred)
        
        # Hessian 损失 (4 项)
        Hess_loss = 0
        if self.Hess_weight > 0:
            Hess_loss = self.hess_loss(y_pred)
        
        # Laplace 损失
        laplace_loss_val = 0
        if self.laplace_weight > 0:
            laplace_loss_val = self.laplace_loss(y_pred)
        
        # L1 损失
        l1_loss_val = 0
        if self.l1_rate > 0:
            l1_loss_val = y_pred.abs().mean()
        
        total_loss = (psf_loss + 
                     self.TV_weight * TV_loss + 
                     self.Hess_weight * Hess_loss +
                     self.laplace_weight * laplace_loss_val +
                     self.l1_rate * l1_loss_val)
        
        return total_loss, psf_loss, TV_loss, Hess_loss

## 6. 数据加载 (对齐官方)

In [14]:
class TwoStageDataset(Dataset):
    """
    两阶段训练数据集 (简化版：单张图像 + 伪噪声)
    
    输入：添加噪声的图像 (带 insert_xy padding)
    目标：原始图像 (用于去噪和反卷积监督)
    """
    def __init__(self, data_path, input_y=128, input_x=128, 
                 insert_xy=16, n_samples=2000):
        self.img_files = sorted(glob.glob(os.path.join(data_path, "*.tif")))
        self.input_y = input_y
        self.input_x = input_x
        self.insert_xy = insert_xy
        self.n_samples = n_samples
        
        # 如果没有找到文件，使用模拟数据
        if len(self.img_files) == 0:
            print(f"Warning: No images found in {data_path}, using synthetic data")
            self.use_synthetic = True
        else:
            self.use_synthetic = False
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        if self.use_synthetic or len(self.img_files) == 0:
            # 模拟数据
            img = np.random.rand(self.input_y, self.input_x).astype(np.float32)
            img = percentile_norm(img)
        else:
            # 随机选择图像
            file_idx = np.random.randint(0, len(self.img_files))
            img = load_image(self.img_files[file_idx])
            
            # 调整尺寸
            if img.shape != (self.input_y, self.input_x):
                img = cv2.resize(img, (self.input_x, self.input_y))
        
        # 添加噪声 (模拟输入)
        noisy = img + np.random.normal(0, 0.1, img.shape).astype(np.float32)
        noisy = np.clip(noisy, 0, 1)
        
        # 添加 padding (insert_xy)
        noisy_padded = np.pad(noisy, ((self.insert_xy, self.insert_xy),
                                       (self.insert_xy, self.insert_xy)),
                              mode='constant')
        
        return (
            torch.from_numpy(noisy_padded[None]),  # 噪声输入 (带 padding)
            torch.from_numpy(img[None])            # 原始目标
        )


## 7. 训练配置

In [15]:
# ========== 路径配置 ==========
DATA_DIR = "datasets/Microtubule/"
FOLDER = "train_data"
PSF_PATH = "../datasets/Microtubule/PSF/psf_emLambda525_dxy0.0313_NA1.3.tif"
SAVE_DIR = "./output_two_stage_complete/"

# ========== 模型配置 ==========
CONV_BLOCK_NUM = 4       # UNet 深度
CONV_NUM = 3             # 每 block 卷积层数
UPSAMPLE_FLAG = True     # 2x 上采样

# ========== 数据配置 ==========
INPUT_Y = 128
INPUT_X = 128
INSERT_XY = 16           # 边界 padding (对齐官方)
BATCH_SIZE = 4
N_SAMPLES = 2000         # 每 iteration 采样数

# ========== 训练配置 ==========
ITERATIONS = 50000       # 总训练 iterations (对齐官方)
START_LR = 5e-5          # 初始学习率
LR_DECAY_FACTOR = 0.5    # 学习率衰减因子
LR_DECAY_INTERVAL = 10000  # 学习率衰减间隔
TEST_INTERVAL = 1000     # 保存 checkpoint 间隔
VALID_INTERVAL = 1000    # 验证间隔

# ========== 损失配置 ==========
MSE_FLAG = False         # False=MAE, True=MSE
DENOISE_LOSS_WEIGHT = 0.5  # 去噪损失权重
HESS_RATE = 0.02         # Hessian 权重
TV_RATE = 0.0            # TV 权重
LAPLACE_RATE = 0.0       # Laplace 权重
L1_RATE = 0.0            # L1 权重

# ========== 其他 ==========
VALID_NUM = 3            # 验证样本数
SEED = 42

print("=" * 60)
print("两阶段训练配置 (完全对齐官方 TensorFlow)")
print("=" * 60)
print(f"数据：{DATA_DIR}{FOLDER}")
print(f"PSF: {PSF_PATH}")
print(f"输出：{SAVE_DIR}")
print(f"模型：depth={CONV_BLOCK_NUM}, n_conv={CONV_NUM}, upsample={UPSAMPLE_FLAG}")
print(f"训练：{ITERATIONS} iterations, batch_size={BATCH_SIZE}")
print(f"学习率：{START_LR}, decay={LR_DECAY_FACTOR}/{LR_DECAY_INTERVAL}")
print(f"损失：denoise_weight={DENOISE_LOSS_WEIGHT}, Hess={HESS_RATE}, TV={TV_RATE}")
print(f"边界处理：insert_xy={INSERT_XY}")
print("=" * 60)

两阶段训练配置 (完全对齐官方 TensorFlow)
数据：datasets/Microtubule/train_data
PSF: ../datasets/Microtubule/PSF/psf_emLambda525_dxy0.0313_NA1.3.tif
输出：./output_two_stage_complete/
模型：depth=4, n_conv=3, upsample=True
训练：50000 iterations, batch_size=4
学习率：5e-05, decay=0.5/10000
损失：denoise_weight=0.5, Hess=0.02, TV=0.0
边界处理：insert_xy=16


## 8. 初始化训练

In [16]:
set_seed(SEED)
ensure_dir(SAVE_DIR)
ensure_dir(os.path.join(SAVE_DIR, 'TrainSampled'))
ensure_dir(os.path.join(SAVE_DIR, 'TestSampled'))

# 保存配置
with open(os.path.join(SAVE_DIR, "config.txt"), "w") as f:
    f.write(f"DATA_DIR: {DATA_DIR}\n")
    f.write(f"FOLDER: {FOLDER}\n")
    f.write(f"PSF_PATH: {PSF_PATH}\n")
    f.write(f"CONV_BLOCK_NUM: {CONV_BLOCK_NUM}\n")
    f.write(f"CONV_NUM: {CONV_NUM}\n")
    f.write(f"UPSAMPLE_FLAG: {UPSAMPLE_FLAG}\n")
    f.write(f"INPUT_Y: {INPUT_Y}, INPUT_X: {INPUT_X}\n")
    f.write(f"INSERT_XY: {INSERT_XY}\n")
    f.write(f"BATCH_SIZE: {BATCH_SIZE}\n")
    f.write(f"ITERATIONS: {ITERATIONS}\n")
    f.write(f"START_LR: {START_LR}\n")
    f.write(f"DENOISE_LOSS_WEIGHT: {DENOISE_LOSS_WEIGHT}\n")
    f.write(f"HESS_RATE: {HESS_RATE}, TV_RATE: {TV_RATE}\n")
    f.write(f"LAPLACE_RATE: {LAPLACE_RATE}, L1_RATE: {L1_RATE}\n")

# 加载 PSF
print(f"加载 PSF: {PSF_PATH}")
psf = make_psf_tensor(PSF_PATH, size=25, device=device)
print(f"PSF 形状：{psf.shape}, 总和：{psf.sum():.6f}")

# 保存 PSF
psf_np = psf.squeeze().cpu().numpy()
save_image(os.path.join(SAVE_DIR, 'psf.tif'), psf_np)

# 初始化模型
print("初始化两阶段 UNet...")
model = TwoStageUNet(
    base_ch=32,
    depth=CONV_BLOCK_NUM,
    n_conv=CONV_NUM,
    upsample=UPSAMPLE_FLAG
).to(device)

print(f"模型参数量：{sum(p.numel() for p in model.parameters()):,}")

# 初始化损失函数
criterion = PSFLoss(
    psf,
    TV_weight=TV_RATE,
    Hess_weight=HESS_RATE,
    laplace_weight=LAPLACE_RATE,
    l1_rate=L1_RATE,
    mse_flag=MSE_FLAG,
    upsample_flag=UPSAMPLE_FLAG,
    insert_xy=INSERT_XY,
    deconv_flag=True
).to(device)

# 初始化优化器
optimizer = torch.optim.Adam(model.parameters(), lr=START_LR, betas=(0.9, 0.999))

# 学习率调度器
def lr_lambda(iteration):
    if (iteration + 1) % LR_DECAY_INTERVAL == 0:
        return LR_DECAY_FACTOR
    return 1.0

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# TensorBoard
writer = SummaryWriter(os.path.join(SAVE_DIR, 'graph'))

# 加载验证数据
print("加载验证数据...")
images_path = os.path.join(DATA_DIR, FOLDER, 'input')
gt_path = os.path.join(DATA_DIR, FOLDER, 'gt')

# 注意：如果没有 gt 目录，使用原图作为 gt
if not os.path.exists(gt_path):
    gt_path = os.path.join(DATA_DIR, FOLDER, 'train_data')
    if not os.path.exists(gt_path):
        gt_path = images_path  # 使用 input 作为 gt

valid_dataset = TwoStageDataset(
    images_path, INPUT_Y, INPUT_X, INSERT_XY, n_samples=VALID_NUM
    images_path, INPUT_Y, INPUT_X, INSERT_XY, n_samples=BATCH_SIZE
)
valid_loader = DataLoader(valid_dataset, batch_size=VALID_NUM, shuffle=False)
# 修改：只返回输入，目标在训练中使用
valid_loader = DataLoader(valid_dataset, batch_size=VALID_NUM, shuffle=False)
input_valid_batch, gt_valid = next(iter(valid_loader))
input_valid = input_valid_batch

# 保存验证样本
print("保存验证样本...")
for i in range(VALID_NUM):
    img = input_valid[i, 0].cpu().numpy()
    img = img[INSERT_XY:INSERT_XY+INPUT_Y, INSERT_XY:INSERT_XY+INPUT_X]
    save_image(os.path.join(SAVE_DIR, 'TrainSampled', f'input_sample_{i}.tif'), img)
    
    gt = gt_valid[i, 0].cpu().numpy()
    save_image(os.path.join(SAVE_DIR, 'TrainSampled', f'gt_sample_{i}.tif'), gt)

print("初始化完成！")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (136145087.py, line 82)

## 9. 训练循环 (完全对齐官方)

In [ ]:
def validate(iteration):
    """验证 (对齐官方 Validate 函数)"""
    print('Validating...')
    valid_start = time.time()
    
    model.eval()
    with torch.no_grad():
        denoised, deconv = model(input_valid)
        
        for i in range(VALID_NUM):
            # 去噪输出
            denoise_out = denoised[i, 0].cpu().numpy()
            denoise_out = percentile_norm(denoise_out)
            save_image(os.path.join(SAVE_DIR, 'TrainSampled', 
                    f'{i}_denoised_iter{iteration:05d}.tif'), denoise_out)
            
            # 反卷积输出
            deconv_out = deconv[i, 0].cpu().numpy()
            if UPSAMPLE_FLAG:
                deconv_out = deconv_out[2*INSERT_XY:2*(INSERT_XY+INPUT_Y),
                                       2*INSERT_XY:2*(INSERT_XY+INPUT_X)]
            else:
                deconv_out = deconv_out[INSERT_XY:INSERT_XY+INPUT_Y,
                                       INSERT_XY:INSERT_XY+INPUT_X]
            deconv_out = percentile_norm(deconv_out)
            save_image(os.path.join(SAVE_DIR, 'TrainSampled',
                    f'{i}_deconved_iter{iteration:05d}.tif'), deconv_out)
    
    model.train()
    elapsed = time.time() - valid_start
    print(f'Validation time: {elapsed:.2f}s')


def save_checkpoint(iteration):
    """保存检查点"""
    checkpoint_path = os.path.join(SAVE_DIR, f'weights_{iteration}.pth')
    torch.save({
        'model': model.state_dict(),
        'iteration': iteration,
    }, checkpoint_path)
    print(f'Saved checkpoint: {checkpoint_path}')


def write_logs(iteration, loss_denoise, loss_deconv):
    """写入 TensorBoard 日志"""
    lr = optimizer.param_groups[0]['lr']
    writer.add_scalar('lr', lr, iteration)
    writer.add_scalar('denoise_loss', loss_denoise, iteration)
    writer.add_scalar('deconv_loss', loss_deconv, iteration)


# 训练
print("=" * 60)
print(f"开始训练，{ITERATIONS} iterations")
print("=" * 60)

model.train()
start_time = time.time()

loss_record_denoise = []
loss_record_deconv = []

for iteration in range(1, ITERATIONS + 1):
    # 加载数据
    dataset = TwoStageDataset(
        images_path, gt_path, INPUT_Y, INPUT_X, INSERT_XY, n_samples=BATCH_SIZE
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    input_g, gt_g = next(iter(loader))
    input_g = input_g.to(device)
    gt_g = gt_g.to(device)
    
    # 前向传播
    denoised, deconv = model(input_g)
    
    # 计算损失
    loss_denoise, _, _, _ = criterion(gt_g, denoised)
    loss_deconv, psf_loss, tv_loss, hess_loss = criterion(gt_g, deconv)
    
    # 加权总损失
    loss = (DENOISE_LOSS_WEIGHT * loss_denoise + 
           (1 - DENOISE_LOSS_WEIGHT) * loss_deconv)
    
    # 反向传播
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    scheduler.step()
    
    # 记录损失
    loss_record_denoise.append(loss_denoise.item())
    loss_record_deconv.append(loss_deconv.item())
    
    # 打印进度
    elapsed = time.time() - start_time
    print(f"{iteration} it | time: {elapsed:.1f}s | " +
          f"denoise_loss = {loss_denoise.item():.3e} | " +
          f"deconv_loss = {loss_deconv.item():.3e}")
    
    # 验证
    if iteration % VALID_INTERVAL == 0:
        validate(iteration)
    
    # 保存 checkpoint 和日志
    if iteration % TEST_INTERVAL == 0:
        save_checkpoint(iteration)
        
        mean_denoise = np.mean(loss_record_denoise)
        mean_deconv = np.mean(loss_record_deconv)
        write_logs(iteration, mean_denoise, mean_deconv)
        
        loss_record_denoise = []
        loss_record_deconv = []
        
        print('Testing...')
        # 这里可以添加测试代码

writer.close()
print("\n训练完成！")

## 10. 绘制损失曲线

In [ ]:
# 从 TensorBoard 读取日志
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

event_acc = EventAccumulator(os.path.join(SAVE_DIR, 'graph'))
event_acc.Reload()

# 提取损失数据
_, denoise_steps, denoise_values = zip(*event_acc.Scalars('denoise_loss'))
_, deconv_steps, deconv_values = zip(*event_acc.Scalars('deconv_loss'))

# 绘制
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(denoise_steps, denoise_values, 'b-', linewidth=2, label='Denoise Loss')
plt.plot(deconv_steps, deconv_values, 'r-', linewidth=2, label='Deconv Loss')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.semilogy(denoise_steps, denoise_values, 'b-', linewidth=2, label='Denoise Loss')
plt.semilogy(deconv_steps, deconv_values, 'r-', linewidth=2, label='Deconv Loss')
plt.xlabel('Iteration')
plt.ylabel('Loss (log scale)')
plt.title('Training Loss (Log Scale)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'loss_curve.png'), dpi=150)
plt.show()

print(f"损失曲线已保存到：{os.path.join(SAVE_DIR, 'loss_curve.png')}")

## 11. 测试推理

In [ ]:
@torch.no_grad()
def infer_two_stage(image_path, checkpoint_path, out_dir=None):
    """
    两阶段推理
    
    Args:
        image_path: 输入图像路径
        checkpoint_path: 模型检查点路径
        out_dir: 输出目录
    
    Returns:
        denoised_path, deconv_path, denoised_np, deconv_np
    """
    if out_dir is None:
        out_dir = SAVE_DIR
    ensure_dir(out_dir)
    
    # 加载模型
    print(f"加载模型：{checkpoint_path}")
    ckpt = torch.load(checkpoint_path, map_location=device)
    
    model = TwoStageUNet(base_ch=32, depth=CONV_BLOCK_NUM, n_conv=CONV_NUM, 
                        upsample=UPSAMPLE_FLAG).to(device)
    model.load_state_dict(ckpt['model'])
    model.eval()
    
    # 加载图像
    print(f"处理图像：{image_path}")
    img = percentile_norm(load_image(image_path))
    
    # 添加 padding
    img_padded = np.pad(img, ((INSERT_XY, INSERT_XY), (INSERT_XY, INSERT_XY)),
                       mode='constant')
    x = torch.from_numpy(img_padded[None, None]).float().to(device)
    
    # 推理
    denoised, deconv = model(x)
    
    # 去噪输出
    denoised_np = denoised[0, 0].cpu().numpy()
    denoised_np = denoised_np[INSERT_XY:INSERT_XY+img.shape[0],
                              INSERT_XY:INSERT_XY+img.shape[1]]
    
    # 反卷积输出
    deconv_np = deconv[0, 0].cpu().numpy()
    if UPSAMPLE_FLAG:
        deconv_np = deconv_np[2*INSERT_XY:2*(INSERT_XY+img.shape[0]),
                              2*INSERT_XY:2*(INSERT_XY+img.shape[1])]
    else:
        deconv_np = deconv_np[INSERT_XY:INSERT_XY+img.shape[0],
                              INSERT_XY:INSERT_XY+img.shape[1]]
    
    # 保存
    name = Path(image_path).stem
    denoised_path = os.path.join(out_dir, f"{name}_denoised.tif")
    deconv_path = os.path.join(out_dir, f"{name}_deconved.tif")
    
    save_image(denoised_path, percentile_norm(denoised_np))
    save_image(deconv_path, percentile_norm(deconv_np))
    
    print(f"去噪结果：min={denoised_np.min():.4f}, max={denoised_np.max():.4f}")
    print(f"反卷积结果：min={deconv_np.min():.4f}, max={deconv_np.max():.4f}")
    print(f"已保存：{denoised_path}, {deconv_path}")
    
    # 可视化
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.imshow(img, cmap="gray")
    plt.title("Input")
    plt.axis("off")
    
    plt.subplot(1, 3, 2)
    plt.imshow(percentile_norm(denoised_np), cmap="gray")
    plt.title("Denoised")
    plt.axis("off")
    
    plt.subplot(1, 3, 3)
    plt.imshow(percentile_norm(deconv_np), cmap="gray")
    plt.title("Deconvolved")
    plt.axis("off")
    
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{name}_result.png"), dpi=150)
    plt.show()
    
    return denoised_path, deconv_path, denoised_np, deconv_np


# 使用最新检查点进行推理
latest_checkpoint = sorted(glob.glob(os.path.join(SAVE_DIR, 'weights_*.pth')))[-1]
test_image = os.path.join(DATA_DIR, FOLDER, '01.tif')

if os.path.exists(test_image):
    denoised_path, deconv_path, denoised_np, deconv_np = infer_two_stage(
        image_path=test_image,
        checkpoint_path=latest_checkpoint,
        out_dir=SAVE_DIR,
    )
else:
    print(f"测试图像不存在：{test_image}")
    print("请使用 infer_two_stage 函数手动测试")

## 12. 结果对比

In [ ]:
# 加载并对比结果
if os.path.exists(test_image):
    original = load_image(test_image)
    original_norm = percentile_norm(original)
    denoise_result_norm = percentile_norm(denoised_np)
    deconv_result_norm = percentile_norm(deconv_np)
    
    plt.figure(figsize=(18, 6))
    
    plt.subplot(1, 4, 1)
    plt.imshow(original_norm, cmap="gray")
    plt.title(f"Original ({original.shape[0]}x{original.shape[1]})")
    plt.axis("off")
    
    plt.subplot(1, 4, 2)
    plt.imshow(denoise_result_norm, cmap="gray")
    plt.title(f"Denoised")
    plt.axis("off")
    
    plt.subplot(1, 4, 3)
    plt.imshow(deconv_result_norm, cmap="gray")
    plt.title(f"Deconvolved ({deconv_np.shape[0]}x{deconv_np.shape[1]})")
    plt.axis("off")
    
    # 剖面线对比
    center_y = original.shape[0] // 2
    x = np.arange(original.shape[1])
    plt.subplot(1, 4, 4)
    plt.plot(x, original_norm[center_y, :], 'b-', label='Original', alpha=0.7)
    plt.plot(x, denoise_result_norm[center_y, :], 'g-', label='Denoised', alpha=0.7)
    
    if deconv_np.shape[0] == original.shape[0] * 2:
        center_y_deconv = center_y * 2
        x_deconv = np.arange(deconv_np.shape[1])
        plt.plot(x_deconv, deconv_result_norm[center_y_deconv, :], 'r-', 
                label='Deconvolved (2x)', alpha=0.7)
    else:
        plt.plot(x, deconv_result_norm[center_y, :], 'r-', label='Deconvolved', alpha=0.7)
    
    plt.xlabel("Pixel")
    plt.ylabel("Normalized Intensity")
    plt.title(f"Horizontal Profile (y={center_y})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "final_comparison.png"), dpi=150, bbox_inches="tight")
    plt.show()
    
    print(f"\n所有结果已保存到：{SAVE_DIR}")
    print(f"- 去噪图像：{denoised_path}")
    print(f"- 反卷积结果：{deconv_path}")
    print(f"- 对比图：{os.path.join(SAVE_DIR, 'final_comparison.png')}")